# Intro
This notebook will build a GCN  + GRU model to predict station demand at timestep T. This will be based off a flow based adjacency between stations to construct our adjacency matrix. 

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, subprocess

WORK_DIR = '/content/drive/MyDrive/DL_Project'
os.chdir(WORK_DIR)
print('cwd:', os.getcwd())

Mounted at /content/drive
cwd: /content/drive/MyDrive/DL_Project


In [2]:
%pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.3 MB/s eta 0:00:00


In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # make cuda error synchonous

In [4]:
import polars as pl
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


from torch_geometric.nn.conv import GCNConv


# Data readin

In [5]:
# read in cabi data
cabi = pl.read_parquet('data/cabi_combined_data.parquet').with_columns([
    pl.col("start_station_id").cast(pl.Int64),
    pl.col("end_station_id").cast(pl.Int64),
])

In [6]:
cabi.head(10)

ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,_source_zip,_source_file
str,str,str,str,str,i64,str,i64,f64,f64,f64,f64,str,str,str
"""1B4445D591115BD9""","""classic_bike""","""2022-01-06 18:39:28""","""2022-01-06 18:43:03""","""Monroe Ave & Leslie Ave""",31087,"""Potomac Ave & Main Line Blvd""",31910,38.820932,-77.053096,38.822738,-77.049265,"""member""","""202201-capitalbikeshare-tripda…","""202201-capitalbikeshare-tripda…"
"""7F4A0E2F03EADEB7""","""classic_bike""","""2022-01-31 19:21:22""","""2022-01-31 19:27:33""","""14th & L St NW""",31283,"""10th & G St NW""",31274,38.903658,-77.031737,38.898243,-77.026235,"""member""","""202201-capitalbikeshare-tripda…","""202201-capitalbikeshare-tripda…"
"""30DD8A84164843AD""","""classic_bike""","""2022-01-07 15:28:39""","""2022-01-07 15:31:01""","""14th & L St NW""",31283,"""12th & L St NW""",31251,38.903658,-77.031737,38.903819,-77.0284,"""member""","""202201-capitalbikeshare-tripda…","""202201-capitalbikeshare-tripda…"
"""FC67665D7682D0A6""","""classic_bike""","""2022-01-27 20:09:25""","""2022-01-27 20:37:02""","""New York Ave & Hecht Ave NE""",31518,"""Nannie Helen Burroughs & Minne…",31704,38.915604,-76.983683,38.901385,-76.941877,"""casual""","""202201-capitalbikeshare-tripda…","""202201-capitalbikeshare-tripda…"
"""7854F7CC4F631A1E""","""classic_bike""","""2022-01-07 16:14:28""","""2022-01-07 16:16:13""","""Falls Church City Hall / Park …",32608,"""Pennsylvania Ave & Park Ave""",32603,38.885434,-77.173605,38.887403,-77.176992,"""member""","""202201-capitalbikeshare-tripda…","""202201-capitalbikeshare-tripda…"
"""FBDCB4296C2A0B2E""","""classic_bike""","""2022-01-24 19:50:35""","""2022-01-24 20:07:00""","""14th & L St NW""",31283,"""Park Rd & Holmead Pl NW""",31602,38.903658,-77.031737,38.9308,-77.0315,"""casual""","""202201-capitalbikeshare-tripda…","""202201-capitalbikeshare-tripda…"
"""6B14178CB767703A""","""classic_bike""","""2022-01-26 19:37:48""","""2022-01-26 19:48:50""","""Adams Mill & Columbia Rd NW""",31104,"""3rd & Elm St NW""",31118,38.922925,-77.042581,38.917622,-77.01597,"""casual""","""202201-capitalbikeshare-tripda…","""202201-capitalbikeshare-tripda…"
"""390C77E802B77D91""","""classic_bike""","""2022-01-15 10:12:52""","""2022-01-15 10:22:15""","""14th & Harvard St NW""",31105,"""11th & S St NW""",31280,38.9268,-77.0322,38.913761,-77.027025,"""member""","""202201-capitalbikeshare-tripda…","""202201-capitalbikeshare-tripda…"
"""131937550248E68C""","""electric_bike""","""2022-01-08 11:48:59""","""2022-01-08 11:56:21""","""14th & Harvard St NW""",31105,"""Calvert St & Woodley Pl NW""",31121,38.926775,-77.032137,38.923583,-77.050046,"""casual""","""202201-capitalbikeshare-tripda…","""202201-capitalbikeshare-tripda…"


# Construct Flow Adjacency

In [7]:
# convert these to int since the code below won't work otherwise
cabi = cabi.with_columns([
    pl.col("start_station_id").cast(pl.Int64),
    pl.col("end_station_id").cast(pl.Int64)
])

In [8]:
# compute bidirectional step counts between stations

flow_df = cabi.filter(pl.col("start_station_id").is_not_null() &
            pl.col("end_station_id").is_not_null() &
            pl.col("start_station_id") != pl.col("end_station_id")
            ).select(["start_station_id", "end_station_id"]
            ).with_columns([
                # compute as directed edge
               pl.min_horizontal("start_station_id", "end_station_id").alias("source"),
                pl.max_horizontal("start_station_id", "end_station_id").alias("dest")
            ]).group_by(["source", "dest"]).agg(pl.len().alias("flow"))


In [ ]:
# make adjacency list symmetric
forward = flow_df.rename({"source": "station", "dest": "neighbor"})
backward = flow_df.rename({"source": "station", "dest": "neighbor"})

adj = pl.concat([forward, backward])

In [10]:
# filter for only top 5 neighbors or top k
K = 5
top_k = adj.sort("flow", descending=True
                 ).group_by("station"
                ).head(K
                ).sort(
                ["station", "flow"],
                descending=[False, True]
                )

In [11]:
# and delete our original data
cabi = cabi.clear()
import gc
gc.collect()

0

# Data Prep
Now that we have tha adjacency matrix, lets pump that into our nodes with our main dataset as well

In [12]:
cabi_demand = pl.read_parquet("data/cabi_master.parquet").with_columns([
    pl.col("start_station_id").cast(pl.Int64),
])

In [13]:
cabi_demand.head(5)

start_station_id,year,started_at_15,demand,latitude,longitude,n_nearby_stations,n_bus_stops,n_metro_stations,n_commuter_bus,n_tourist_bus,n_carshare,n_hotels,n_museums,n_recreation,n_shopping,n_universities,n_schools,n_schools_private,n_parks,n_national_parks,m5_4_bike_,m5_5_sidew,m8_3_trail,m8_4_land_,m8_5_posit,m8_6_flood,m9_1_vacan,m9_2_stree,m9_3_polic,m9_4_fire_,m9_5_HIN,m6_5_resta,m6_6_liquo,m8_1_urban,total_popE,pop_maleE,pop_femaleE,median_ageE,median_hh_incomeE,in_low_stress_bikeshed,in_total_bikeshed,temp_f,precip_in,hour,minute,day_of_week,month,week_of_year,is_weekend,day_of_week_sin,day_of_week_cos,week_of_year_sin,week_of_year_cos,date,is_holiday
i64,f64,"datetime[μs, UTC]",i32,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64,i8,i8,i8,i8,i8,i8,f64,f64,f64,f64,date,i8
22907,2024.0,2026-01-15 17:00:00 UTC,0,38.918767,-76.965328,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.005958,0.993762,0.241886,0.94982,0.793049,0.869033,0.956766,0.22045,0.46039,0.389143,0.858014,0.36368,0.233751,0.388847,5028.0,2653.0,2375.0,41.9,80536.0,0,1,28.9,0.0,17,0,4,1,3,0,-0.433884,-0.900969,0.354605,0.935016,2026-01-15,0
22907,2024.0,2026-01-15 17:15:00 UTC,0,38.918767,-76.965328,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.005958,0.993762,0.241886,0.94982,0.793049,0.869033,0.956766,0.22045,0.46039,0.389143,0.858014,0.36368,0.233751,0.388847,5028.0,2653.0,2375.0,41.9,80536.0,0,1,28.9,0.0,17,15,4,1,3,0,-0.433884,-0.900969,0.354605,0.935016,2026-01-15,0
22907,2024.0,2026-01-15 17:45:00 UTC,0,38.918767,-76.965328,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.005958,0.993762,0.241886,0.94982,0.793049,0.869033,0.956766,0.22045,0.46039,0.389143,0.858014,0.36368,0.233751,0.388847,5028.0,2653.0,2375.0,41.9,80536.0,0,1,28.9,0.0,17,45,4,1,3,0,-0.433884,-0.900969,0.354605,0.935016,2026-01-15,0
22907,2024.0,2026-01-15 18:00:00 UTC,0,38.918767,-76.965328,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.005958,0.993762,0.241886,0.94982,0.793049,0.869033,0.956766,0.22045,0.46039,0.389143,0.858014,0.36368,0.233751,0.388847,5028.0,2653.0,2375.0,41.9,80536.0,0,1,26.5,0.0,18,0,4,1,3,0,-0.433884,-0.900969,0.354605,0.935016,2026-01-15,0
22907,2024.0,2026-01-15 18:15:00 UTC,0,38.918767,-76.965328,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.005958,0.993762,0.241886,0.94982,0.793049,0.869033,0.956766,0.22045,0.46039,0.389143,0.858014,0.36368,0.233751,0.388847,5028.0,2653.0,2375.0,41.9,80536.0,0,1,26.5,0.0,18,15,4,1,3,0,-0.433884,-0.900969,0.354605,0.935016,2026-01-15,0


## Build Edge Index
A GNN is made of nodes and edges, we can use our adjacency matrix as our weighted edge index.

In [14]:
# get all station ids
# need to convert to integer since we did that for everything before
station_ids = cabi_demand.with_columns([pl.col("start_station_id").cast(pl.Int64)]
                                ).select("start_station_id"
                                 ).unique(
                                ).sort("start_station_id"
                                ).get_column("start_station_id"
                                ).to_list()
                            

# convert to an integer index
station_idx_mapper = {si: i for i, si in enumerate(station_ids)}
N = len(station_ids)
# map station ids to the stations / neighors
edges = top_k.with_columns([
    pl.col("station").cast(pl.Int64),
    pl.col("neighbor").cast(pl.Int64),
]).with_columns([
     pl.col("station").map_elements(lambda x: station_idx_mapper.get(x, None), return_dtype=pl.Int64).alias("src_idx"),
    pl.col("neighbor").map_elements(lambda x: station_idx_mapper.get(x, None), return_dtype=pl.Int64).alias("dst_idx"),
]).select(["src_idx", "dst_idx", "flow"]).drop_nulls()
# convert to torch tensors
# get sources
source = torch.tensor(edges["src_idx"].to_list(), dtype=torch.long)
#get destinations
dst = torch.tensor(edges["dst_idx"].to_list(), dtype=torch.long)
# combine into a complete stacked edge index
edge_index = torch.stack([source, dst], dim=0) 

# Normalize weights from 0 to 1
flow = torch.tensor(edges["flow"].to_list(), dtype=torch.float)
edge_data = (flow / flow.max()).unsqueeze(1) 
print("edge data shape")
print(edge_data.shape)
print("edge index shape")
print(edge_index.shape)



edge data shape
torch.Size([1993, 1])
edge index shape
torch.Size([2, 1993])


## Build Feature Tensor from cabi_demand
As we get into this we need to pivot some of our features to work with this new thing. We will first want to split up our station specific featurs from our time based ones.This will help later in our modeling.

In [ ]:
# Establish our 2 feature sets
static_feat = [
    "n_bus_stops", "n_metro_stations", "n_commuter_bus", "n_carshare",
    "n_hotels", "n_parks", "n_shopping", "total_popE", "median_hh_incomeE",
    "in_low_stress_bikeshed", "latitude", "longitude",
    "m5_4_bike_", "m5_5_sidew", "m8_3_trail", "m8_1_urban",
]
# time based, this needs to only be repeated on our temporal layer
time_feat = [
    "temp_f", "precip_in",
    "hour", "minute",
    "day_of_week_sin", "day_of_week_cos",
    "week_of_year_sin", "week_of_year_cos",
    "is_weekend", "is_holiday",
    "month",
]

all_feat = static_feat + time_feat


In [16]:
# lets sort our timesteps 
distinct_timesteps = cabi_demand.select("started_at_15"
                                    ).unique().sort("started_at_15"
                                    ).get_column("started_at_15").to_list()
T = len(distinct_timesteps)
# get our mapper here as well
timestep_mapper = {t: i for i, t in enumerate(distinct_timesteps)}

In [17]:
# Pivot over from wide to long dense arrays
# First make empty zero arrays for missing timesteps and such
F = len(all_feat)
x = np.zeros((N, T, F), dtype=np.float32)
y = np.zeros((N,T), dtype=np.float32)
# select columns and add index
# this needs to be an integer).with_columns([
demand_selected = cabi_demand.with_columns(
    [pl.col("start_station_id").cast(pl.Int64)]
    ).with_columns([
        pl.col("start_station_id").replace(station_idx_mapper).cast(pl.Int64).alias("node_idx"),
        pl.col("started_at_15").replace(timestep_mapper).cast(pl.Int64).alias("time_idx"),
]).select(["node_idx", "time_idx", "demand"] + all_feat
          ).drop_nulls() # also drop any nulls  we see

# extract our indexs and values for them
node_idx = demand_selected["node_idx"].to_numpy().astype(np.int64)
time_idx = demand_selected["time_idx"].to_numpy().astype(np.int64)
feat_vals = demand_selected.select(all_feat).to_numpy()
demand_vals = demand_selected["demand"].to_numpy()

# to tensors!

x[node_idx, time_idx, :] = feat_vals
y[node_idx, time_idx] = demand_vals
x = torch.tensor(x, dtype=torch.float)
y = torch.tensor(y, dtype = torch.float)


In [ ]:
# check shapes
print("x shape")
print(x.shape)
print("y shape")
print(y.shape)

x shape
torch.Size([419, 148272, 27])
y shape
torch.Size([419, 148272])


## Window Slicing
We need to have sliding windows for our model to draw on here. This is because the model will only consume a set length of data at a time. It will then use this sequence to predict ahead a certain X amount of steps.

In [ ]:
# instead of a simple loop we need to create this in a dataloader
# we initially tried a loop, but due to the size of the data it runs us out of memory
# instead we can create a dataset that slices itself

class WindowDataset(Dataset):
    def __init__(self, x, y, seq_len, horizon):
        self.x = x
        self.y = y
        self.seq_len = seq_len
        self.horizon = horizon
        self.n_windows = x.shape[1] - seq_len - horizon + 1

    def __len__(self):
        return self.n_windows

    def __getitem__(self, idx):
        x_win = self.x[:, idx : idx + self.seq_len, :]      
        y_win = self.y[:, idx + self.seq_len : idx + self.seq_len + self.horizon]  
        return x_win, y_win
    

## Train Test val Split
We defined our window dataset, lets handle the splitting and data loading here. Note we have to sequentially split our data, we can't do a normal train test split so that we avoid data leakage and keep a squential order for our GRU to understand.

In [ ]:
# define our seq len and horizon
seq_len = 8
predict_horizon = 4

# establish length of each segment
# weare doing 70% train, 15% val, and 15% test
T = x.shape[1]
train_end = int(T * 0.70)
val_end   = int(T * 0.85)

#build datasets for train / val / test
train_dataset = WindowDataset(x[:, :train_end, :], y[:, :train_end],seq_len, predict_horizon)
val_dataset   = WindowDataset(x[:, train_end:val_end, :], y[:, train_end:val_end], seq_len, predict_horizon)
test_dataset  = WindowDataset(x[:, val_end:, :], y[:, val_end:], seq_len, predict_horizon)

# load them into dataloaders
# note, we don't shuffle anything in order to keep chronological order.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

# Modeling
Now that we have our data prepped it's time for the main event. 

The general architecture

Input --> GCNConv Layer --> GRU Layer --> Linear Layer

In [21]:

# https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.nn.conv.GCNConv.html?highlight=gcnconv#torch_geometric.nn.conv.GCNConv

class FlowGCN_GRU(nn.Module):
    def __init__(self, in_features, hidden_dim, horizon, num_nodes):
        # init the main class
        super().__init__()
        ## INPUTS ##
        self.in_features = in_features
        self.hidden_dim = hidden_dim
        self.horizon = horizon
        self.num_nodes = num_nodes
        
        ## MODEL LAYERS ##
        # establish gcn model
        self.gcn = GCNConv(in_features, hidden_dim, add_self_loops=False)
        # establish gru
        self.gru = nn.GRU(input_size = hidden_dim, hidden_size = hidden_dim, batch_first=True)
        # output layer
        self.output_layer = nn.Linear(hidden_dim, horizon)
        self.relu = nn.ReLU() # need our relu too

    def forward(self, x, edge_index, edge_attr = None):
        # INPUT GUIDE:
        # x = input data, [batch, n, seq_len, f]
        # edge_index = indices of our edges
        # edge_attr = our edge weights , in this case its the groupings
        
        # lets forward this stuff idk
        #### Methodology: ####
        # 1. Extract information for batch_size, N, seq_len, and Features from our x.shape
        # 2. Create output var for graph network
        # 3. Start for loop over the sequence length in time
        #   3.1 get timestep
        #   3.2 Apply GCN
        #   3.3 Relu
        #   3.4 Append results to output var
        # 4. Stack output list into a torch sequence
        # 5. Plug into GRU
        # 6. Plug into final layer
        # 7. structure output and return
        
        ### EXECUTION ###
        # 1. Extract information for batch_size, N, seq_len, and Features from our x.shape
        batch_size, N, seq_len, F = x.shape
        # 2. Create output var for graph network
        gcn_results = []
        # 3. Start for loop over the sequence length in time
        for time in range(seq_len):
            # 3.1 get timestep
            x_t = x[:, :, time, :]
            # reshape since GCN is kinda annoying, it wants it in long format for all nodes
            x_t = x_t.reshape(batch_size * N, F)
            # 3.2 Apply GCN
            gcn_inner_out = self.gcn(x_t, edge_index, edge_attr)
            # 3.3 Relu
            h_t = self.relu(gcn_inner_out)
            # 3.4 Append results to output var
            # also reshape again to work with our hidden dim
            h_t = h_t.reshape(batch_size, N, self.hidden_dim)       
            gcn_results.append(h_t)
            
        # 4. Stack output list into a torch sequence
        gcn_stacked = torch.stack(gcn_results, dim = 2)
        # 5. Plug into GRU
        # reformat so each node is independent
        gcn_stacked = gcn_stacked.reshape(batch_size * N, seq_len, self.hidden_dim)
        gru_out, h_n = self.gru(gcn_stacked) 
        # 6. Plug into final layer
        out = self.output_layer(h_n.squeeze(0))
        # 7. structure output and return
        out = out.reshape(batch_size, N, self.horizon)
        return out

In [22]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# checing that it works
model = FlowGCN_GRU(
    in_features=F, 
    hidden_dim=64,
    horizon=predict_horizon,
    num_nodes=N,
).to(device)
print(model)

FlowGCN_GRU(
  (gcn): GCNConv(27, 64)
  (gru): GRU(64, 64, batch_first=True)
  (output_layer): Linear(in_features=64, out_features=4, bias=True)
  (relu): ReLU()
)


# Training loop
Now that we got the model all figured out lets get out training loop in place.

In [ ]:
# establish our optimizer (adam)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# and just using normal MSELoss,  we may want to tweak this
criterion = nn.MSELoss()
# move static graph to device
edge_index = edge_index.to(device)
edge_data = edge_data.to(device)

In [24]:
print(f"N (num nodes):          {N}")
print(f"max edge index value:   {edge_index.max().item()}")
print(f"min edge index value:   {edge_index.min().item()}")
print(f"edge_index shape:       {edge_index.shape}")
print(f"x shape:                {x.shape}")  # [N, T, F]

# check for any index that would be out of bounds
assert edge_index.max().item() < N, \
    f"edge_index contains index {edge_index.max().item()} but N={N}"

N (num nodes):          419
max edge index value:   418
min edge index value:   0
edge_index shape:       torch.Size([2, 1993])
x shape:                torch.Size([419, 148272, 27])


In [ ]:
# TRAINING LOOP TIME
from tqdm import tqdm # add for training bars
epochs = 20 # might also adjust as needed 
best_val_loss = float("inf") # establish this
epoch_bar = tqdm(range(epochs), desc="Training", unit="epoch")
# Start epoch loop
for epoch in range(epochs):
    # establish model and set loss to 0
    model.train()
    training_loss = 0.0
    # add bar for training
    train_bar = tqdm(train_loader, 
                     desc=f"  Epoch {epoch+1:02d} Train", 
                     leave=False,  # clears after each epoch
                     unit="batch")
    
    # start batches
    for x_b, y_b in train_loader:
        # move to device
        x_b = x_b.to(device)
        y_b = y_b.to(device)
        
        # zero out optimizer
        optimizer.zero_grad()
        # make predicitons
        pred = model(x_b, edge_index, edge_data)
        # calculate loss
        loss = criterion(pred, y_b)
        # update weights
        loss.backward()
        #step forward
        optimizer.step()
        
        training_loss += loss.item()
        
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    
    # average loss over the batches
    training_loss = training_loss / len(train_loader)
    
    # validation
    model.eval()
    validation_loss = 0.0
    # add bar for validation
    val_bar = tqdm(val_loader,
                   desc=f"  Epoch {epoch+1:02d} Val  ",
                   leave=False,
                   unit="batch")
    
    with torch.no_grad():
        for x_b, y_b in val_loader:
            # move to device
            x_b = x_b.to(device)
            y_b = y_b.to(device)
            # make predictions
            pred = model(x_b, edge_index, edge_data)
            loss = criterion(pred, y_b)
            validation_loss += loss.item() # add to loss
            
    validation_loss = validation_loss / len(val_loader)
    
    ## Add cckepoint for the best model
    if validation_loss < best_val_loss:
        best_val_loss = validation_loss
        torch.save(model.state_dict(), "best_model_flow.pt")
        saved = "saved!"
    else:
        saved = ""
    
    # Print outs for each phase
    epoch_bar.set_postfix({
        "train": f"{training_loss:.4f}",
        "val":   f"{validation_loss:.4f}",
        "best":  f"{best_val_loss:.4f}",
        "":      saved,
    })

# print final best value
print(f"Best Val Loss: {best_val_loss:.4f}")

Training:   0%|          | 0/20 [00:00<?, ?epoch/s]